# Face Emotion Detection - AffectNet Training

This notebook downloads AffectNet from Kaggle, reads AffectNet labels correctly, trains a MobileNetV2 emotion classifier, and lets you upload an image for model testing.

Supported AffectNet inputs:
- Kaggle dataset `mstjebashazida/affectnet`.
- AffectNet-style annotation CSV files with an image path column and an expression/label column.
- Folder-based class structure, such as `train/happy/*.jpg` or `neutral/*.png`.

AffectNet numeric expression mapping used here: `0 neutral`, `1 happy`, `2 sad`, `3 surprise`, `4 fear`, `5 disgust`, `6 angry`. The notebook skips `7 contempt`, `8 none`, `9 uncertain`, and `10 non-face`.

## 1. Install Libraries

Run this cell when setting up a new notebook environment. If you are using Kaggle or Colab and TensorFlow is already installed, you can skip it.

In [ ]:
!pip install -q kagglehub tensorflow opencv-python scikit-learn matplotlib pandas pillow

In [ ]:
import os
import json
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import kagglehub
except ImportError:
    kagglehub = None

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Configure Datasets

Add Kaggle dataset slugs to `KAGGLE_DATASETS` when you want the notebook to download data with `kagglehub`.

On Kaggle Notebook, the recommended setup is to use **Add Input / Add Data** and attach the dataset there. The notebook will automatically use folders under `/kaggle/input`, so you do not need to upload `kaggle.json`.

Example Kaggle slugs:
- `msambare/fer2013`
- `jonathanoheix/face-expression-recognition-dataset`

For Colab or a local machine, prepare your Kaggle token only if you want to download through the Kaggle API.


In [ ]:
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD = 8
EPOCHS_FINE_TUNE = 8
VALID_SIZE = 0.15
TEST_SIZE = 0.15

KAGGLE_DATASETS = [
    'mstjebashazida/affectnet'
]

# Optional: set this to local extracted dataset folder(s) to skip Kaggle download.
# Kaggle Notebook users can leave this empty after adding the dataset as an Input;
# the download cell will automatically use folders under /kaggle/input.
# Example local: [r'C:\\datasets\\affectnet']
# Example Colab Drive: ['/content/drive/MyDrive/datasets/affectnet']
# Example Kaggle Input if you want to be explicit: ['/kaggle/input/affectnet']
LOCAL_DATASET_ROOTS = []

# Keep False for large datasets; kagglehub returns a usable local cache path.
COPY_DATASETS_TO_WORKDIR = False

WORK_DIR = Path('emotion_workdir')
RAW_DIR = WORK_DIR / 'raw'
PROCESSED_DIR = WORK_DIR / 'processed'
MODEL_DIR = WORK_DIR / 'models'
UPLOAD_DIR = WORK_DIR / 'uploaded_images'

for path in [RAW_DIR, PROCESSED_DIR, MODEL_DIR, UPLOAD_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Download Datasets from Kaggle

In [ ]:
def is_colab_runtime():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def is_kaggle_runtime():
    return Path('/kaggle/input').exists()


def find_kaggle_input_roots():
    input_dir = Path('/kaggle/input')
    if not input_dir.exists():
        return []

    roots = [p.resolve() for p in input_dir.iterdir() if p.is_dir()]
    return roots


def prepare_kaggle_credentials():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists() or os.environ.get('KAGGLE_USERNAME'):
        return

    if is_kaggle_runtime():
        print('Kaggle credentials not found, but this is a Kaggle runtime. Add the dataset as an Input, or enable internet/API access if you want kagglehub download.')
        return

    if not is_colab_runtime():
        print('Local Kaggle auth not found. Put kaggle.json in ~/.kaggle or set KAGGLE_USERNAME/KAGGLE_KEY.')
        return

    print('Upload kaggle.json from your Kaggle account settings.')
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('kaggle.json was not uploaded.')

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_bytes(uploaded['kaggle.json'])
    try:
        os.chmod(kaggle_json, 0o600)
    except OSError:
        pass
    print('Saved Kaggle credentials to:', kaggle_json)


def download_kaggle_datasets(dataset_slugs, raw_dir, copy_to_workdir=False):
    if LOCAL_DATASET_ROOTS:
        roots = [Path(p).expanduser().resolve() for p in LOCAL_DATASET_ROOTS]
        missing = [str(p) for p in roots if not p.exists()]
        if missing:
            raise FileNotFoundError('LOCAL_DATASET_ROOTS not found: ' + ', '.join(missing))
        print('Using configured dataset roots:', roots)
        return roots

    kaggle_input_roots = find_kaggle_input_roots()
    if kaggle_input_roots:
        print('Using Kaggle Input dataset roots:', kaggle_input_roots)
        return kaggle_input_roots

    if kagglehub is None:
        raise ImportError('kagglehub is not installed. Run: pip install kagglehub')

    prepare_kaggle_credentials()

    downloaded_paths = []
    for slug in dataset_slugs:
        print(f'Downloading or locating Kaggle dataset: {slug}')
        dataset_path = Path(kagglehub.dataset_download(slug)).resolve()

        if copy_to_workdir:
            target_path = raw_dir / slug.replace('/', '__')
            if target_path.exists():
                shutil.rmtree(target_path)
            shutil.copytree(dataset_path, target_path)
            dataset_path = target_path.resolve()

        downloaded_paths.append(dataset_path)
        print('Dataset root:', dataset_path)

    return downloaded_paths


dataset_roots = download_kaggle_datasets(
    KAGGLE_DATASETS,
    RAW_DIR,
    copy_to_workdir=COPY_DATASETS_TO_WORKDIR,
)
dataset_roots


## 4. Normalize Labels and Load Data

This cell scans images from all downloaded datasets, then normalizes label names to the seven shared classes. If a dataset uses different class names, add mappings to `LABEL_ALIASES`.

In [ ]:
CLASS_NAMES = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

AFFECTNET_ID_TO_LABEL = {
    0: 'neutral',
    1: 'happy',
    2: 'sad',
    3: 'surprise',
    4: 'fear',
    5: 'disgust',
    6: 'angry',
    7: None,      # contempt: skipped for 7-class training
    8: None,      # none
    9: None,      # uncertain
    10: None,     # non-face
}

LABEL_ALIASES = {
    'angry': 'angry', 'anger': 'angry', 'angriness': 'angry',
    'disgust': 'disgust', 'disgusted': 'disgust',
    'fear': 'fear', 'fearful': 'fear',
    'happy': 'happy', 'happiness': 'happy',
    'neutral': 'neutral',
    'sad': 'sad', 'sadness': 'sad',
    'surprise': 'surprise', 'surprised': 'surprise',
}

AFFECTNET_LABEL_COLUMNS = [
    'expression', 'exp', 'emotion', 'label', 'class', 'category', 'emotion_label', 'expression_label'
]
AFFECTNET_PATH_COLUMNS = [
    'subDirectory_filePath', 'subdirectory_filepath', 'filepath', 'file_path', 'path', 'image', 'image_path', 'filename', 'file'
]
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def normalize_label(value, numeric_mapping='affectnet'):
    if pd.isna(value):
        return None

    text = str(value).strip().lower().replace(' ', '_')
    if text in {'', 'nan', 'none', 'uncertain', 'non_face', 'non-face', 'contempt'}:
        return None

    try:
        numeric = int(float(text))
        if numeric_mapping == 'affectnet':
            return AFFECTNET_ID_TO_LABEL.get(numeric)
    except ValueError:
        pass

    return LABEL_ALIASES.get(text)


def infer_label_from_path(path):
    parts = [p.lower().replace(' ', '_') for p in Path(path).parts]
    for part in reversed(parts):
        label = normalize_label(part, numeric_mapping='affectnet')
        if label is not None:
            return label
    return None


def find_existing_image(root, value):
    candidate = Path(str(value).strip())
    candidates = []
    if candidate.is_absolute():
        candidates.append(candidate)
    else:
        clean_value = str(value).strip().lstrip('/\\')
        candidates.extend([
            Path(root) / candidate,
            Path(root) / clean_value,
            Path(root) / 'Manually_Annotated' / 'Manually_Annotated_Images' / clean_value,
            Path(root) / 'Automatically_Annotated' / 'Automatically_Annotated_Images' / clean_value,
        ])

    for item in candidates:
        if item.exists() and item.suffix.lower() in IMAGE_EXTS:
            return item
    return None


def first_matching_column(columns, candidates):
    normalized = {str(col).strip().lower(): col for col in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    return None


def collect_affectnet_csv_records(root):
    records = []
    for csv_path in Path(root).rglob('*.csv'):
        try:
            sample = pd.read_csv(csv_path, nrows=5)
        except Exception:
            continue

        path_col = first_matching_column(sample.columns, AFFECTNET_PATH_COLUMNS)
        label_col = first_matching_column(sample.columns, AFFECTNET_LABEL_COLUMNS)
        if path_col is None or label_col is None:
            continue

        print('Reading AffectNet annotations:', csv_path)
        for chunk in pd.read_csv(csv_path, chunksize=50000):
            for _, row in chunk.iterrows():
                label = normalize_label(row[label_col], numeric_mapping='affectnet')
                if label is None:
                    continue
                image_path = find_existing_image(root, row[path_col])
                if image_path is None:
                    continue
                records.append({
                    'path': str(image_path),
                    'label': label,
                    'source': Path(root).name,
                    'annotation_file': csv_path.name,
                })
    return records


def collect_folder_label_records(root):
    records = []
    for file_path in Path(root).rglob('*'):
        if file_path.suffix.lower() not in IMAGE_EXTS:
            continue
        label = infer_label_from_path(file_path.relative_to(root))
        if label is not None:
            records.append({'path': str(file_path), 'label': label, 'source': Path(root).name})
    return records


def collect_image_files(dataset_roots):
    records = []
    for root in dataset_roots:
        root = Path(root)
        csv_records = collect_affectnet_csv_records(root)
        if csv_records:
            records.extend(csv_records)
        else:
            records.extend(collect_folder_label_records(root))

    df = pd.DataFrame(records)
    if df.empty:
        return pd.DataFrame(columns=['path', 'label', 'source'])
    return df.drop_duplicates('path').reset_index(drop=True)


df_images = collect_image_files(dataset_roots)
print('Images found:', len(df_images))
if not df_images.empty:
    display(df_images['label'].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int))
df_images.head()

## 5. Create Train/Validation/Test Split

In [ ]:
if df.empty:
    raise ValueError(
        'No valid AffectNet data found. Check dataset_roots, annotation CSV columns, or folder labels.'
    )

df = df[df['label'].isin(CLASS_NAMES)].copy()
df['label_id'] = df['label'].map({name: idx for idx, name in enumerate(CLASS_NAMES)})

min_class_count = int(df['label'].value_counts().min())
if min_class_count < 2:
    raise ValueError('Each class needs at least 2 samples for stratified train/valid/test split.')

train_df, temp_df = train_test_split(
    df,
    test_size=VALID_SIZE + TEST_SIZE,
    stratify=df['label'],
    random_state=SEED,
)

relative_test = TEST_SIZE / (VALID_SIZE + TEST_SIZE)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test,
    stratify=temp_df['label'],
    random_state=SEED,
)

print(len(train_df), len(valid_df), len(test_df))
display(pd.DataFrame({
    'train': train_df['label'].value_counts(),
    'valid': valid_df['label'].value_counts(),
    'test': test_df['label'].value_counts(),
}).reindex(CLASS_NAMES).fillna(0).astype(int))

## 6. Create TensorFlow Datasets

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def make_dataset(dataframe, training=False):
    paths = dataframe['path'].astype(str).values
    labels = dataframe['label_id'].astype(np.int32).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(dataframe), 10000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df)
test_ds = make_dataset(test_df)

In [ ]:
plt.figure(figsize=(12, 6))
for images, labels in train_ds.take(1):
    for i in range(min(12, images.shape[0])):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(CLASS_NAMES[int(labels[i])])
        plt.axis('off')
plt.tight_layout()

## 7. Build a Transfer Learning Model

The model uses MobileNetV2 for fast training. If you have a GPU and a large dataset, you can switch to EfficientNetB0/B3.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name='augmentation')

base_model = keras.applications.MobileNetV2(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 8. First Training Stage: Freeze the Backbone

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_DIR / 'best_emotion_model.keras',
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
    ),
]

history_head = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks,
)

## 9. Fine-tune the Backbone

In [ ]:
base_model.trainable = True

# Keep the early layers frozen and fine-tune only the later layers.
fine_tune_at = int(len(base_model.layers) * 0.70)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_fine = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_FINE_TUNE,
    callbacks=callbacks,
)

## 10. Evaluate the Model

In [ ]:
best_model = keras.models.load_model(MODEL_DIR / 'best_emotion_model.keras')
test_loss, test_acc = best_model.evaluate(test_ds)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues')
plt.title('Confusion Matrix')
plt.tight_layout()

## 11. Plot Training Curves

In [ ]:
def merge_history(*histories):
    merged = {}
    for hist in histories:
        for key, values in hist.history.items():
            merged.setdefault(key, []).extend(values)
    return merged

hist = merge_history(history_head, history_fine)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist['accuracy'], label='train')
plt.plot(hist['val_accuracy'], label='valid')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist['loss'], label='train')
plt.plot(hist['val_loss'], label='valid')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()

## 12. Save the Model and Metadata

In [ ]:
final_model_path = MODEL_DIR / 'face_emotion_mobilenetv2.keras'
best_model.save(final_model_path)

metadata = {
    'class_names': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'kaggle_datasets': KAGGLE_DATASETS,
    'test_accuracy': float(test_acc),
}

with open(MODEL_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('Saved model:', final_model_path)
print('Saved metadata:', MODEL_DIR / 'metadata.json')

## 13. Upload an Image and Test the Model

In [ ]:
def predict_emotion(image_path, model=best_model):
    image = Image.open(image_path).convert('RGB').resize(IMG_SIZE)
    array = np.asarray(image, dtype=np.float32)[None, ...]
    probs = model.predict(array, verbose=0)[0]
    pred_idx = int(np.argmax(probs))

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'{CLASS_NAMES[pred_idx]} - {probs[pred_idx]:.2%}')
    plt.show()

    return pd.DataFrame({'emotion': CLASS_NAMES, 'probability': probs}).sort_values('probability', ascending=False)


def upload_image_for_prediction(upload_dir=UPLOAD_DIR):
    upload_dir.mkdir(parents=True, exist_ok=True)

    if is_colab_runtime():
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        if not uploaded:
            raise ValueError('No image was uploaded.')
        filename, content = next(iter(uploaded.items()))
        image_path = upload_dir / filename
        image_path.write_bytes(content)
    else:
        image_path = Path(input('Enter image path to test: ')).expanduser()
        if not image_path.exists():
            raise FileNotFoundError(image_path)

    print('Testing image:', image_path)
    return predict_emotion(image_path)


# Run this after training/loading best_model:
# upload_image_for_prediction()

## 14. Detect Faces from an Image/Webcam Frame and Predict Emotions

This section uses OpenCV Haar Cascade to crop faces before prediction. You can replace it with MTCNN or MediaPipe if you need higher accuracy.

In [ ]:
import cv2

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def detect_faces_and_predict(image_path, model=best_model):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        raise FileNotFoundError(image_path)

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))

    for (x, y, w, h) in faces:
        face = bgr[y:y+h, x:x+w]
        face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face_rgb = cv2.resize(face_rgb, IMG_SIZE)
        probs = model.predict(face_rgb[None, ...].astype(np.float32), verbose=0)[0]
        pred = CLASS_NAMES[int(np.argmax(probs))]
        conf = float(np.max(probs))

        cv2.rectangle(bgr, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(bgr, f'{pred}: {conf:.2f}', (x, max(y-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.axis('off')
    plt.show()

    return faces

# Example:
# detect_faces_and_predict('path_to_face_image.jpg')